# Context Engineering: The Architecture of LLM Memory

Context Engineering is the discipline of designing and managing the information fed into a Large Language Model's (LLM) processing window. While Prompt Engineering focuses on instructions, **Context Engineering** focuses on information selection, compression, and priority management. This guide explores the mechanics of memory and the strategies for building reliable, production-grade applications using **LangChain** and **Google Gemini**.

## 1. The Anatomy of Context
Context is the "RAM" (Random Access Memory) of an AI system. It contains the instructions, current conversation, retrieve data, and tool outputs. 

### The Processing Cycle:
1. **Input**: A mixture of prompt templates and dynamic data.
2. **Processing**: The LLM analyzes the entire context simultaneously (Self-Attention).
3. **Output**: The model predicts the next token based on all available information.

Efficiency in this window directly correlates with the quality of the reasoning and the cost of the application.

## 2. Tokenization & Computation
Large Language Models do not read characters or words; they process tokens. Understanding the "Token Economy" is essential for managing window limits and latency.

In [6]:
import os
import json
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage

load_dotenv()

# Define model once to use throughout the guide
model = "gemini-2.5-flash"

# Initialize the primary model instance
llm = ChatGoogleGenerativeAI(model=model)

def analyze_token_usage(text: str):
    # LangChain utility to count tokens based on the current model's tokenizer
    tokens = llm.get_num_tokens(text)
    print(f"Content Length (Characters): {len(text)}")
    print(f"Token Count: {tokens}")
    print(f"Efficiency: {len(text)/tokens:.2f} chars/token")

sample_data = "Context Engineering is the practice of managing the LLM's workspace."
analyze_token_usage(sample_data)

Content Length (Characters): 68
Token Count: 14
Efficiency: 4.86 chars/token


## 3. Decoding & Sampling Mechanics
Decoding strategies control how an LLM selects the next token from its probability cloud. Tuning these parameters is vital for balance between reliability and creativity.

### Critical Parameters:
- **Temperature**: Scales the probability distribution. Lower temperature (0.0) makes the model deterministic; higher temperature (1.0+) increases randomness.
- **Top-P (Nucleus Sampling)**: Selects tokens whose cumulative probability exceeds a threshold **P**. This prevents the model from choosing from a massive list of low-probability words.
- **Top-K**: Limits the selection to the absolute **K** most probable tokens.

In [9]:
def demonstrate_sampling(temperature, top_p, top_k=None):
    sampling_llm = ChatGoogleGenerativeAI(
        model=model,
        temperature=temperature,
        top_p=top_p,
        top_k=top_k
    )
    prompt = "The future of artificial intelligence in healthcare will be..."
    response = sampling_llm.invoke(prompt)
    print(f"Result (Temp={temperature}, Top-P={top_p}):\n{response.content[:150]}...")

# Focused and Precise
demonstrate_sampling(temperature=0, top_p=0.1)

# Creative and Varied
demonstrate_sampling(temperature=1.0, top_p=0.9)



Result (Temp=0, Top-P=0.1):
The future of artificial intelligence in healthcare will be **transformative, collaborative, and deeply integrated**, fundamentally reshaping how we p...
Result (Temp=1.0, Top-P=0.9):
The future of artificial intelligence in healthcare will be **transformative, highly personalized, and data-driven, yet complex and fraught with ethic...


## 4. Physical Limits & Focus Phenomena
Every model has a finite **Context Window**. While models like Gemini 2.5 Pro offer massive windows (up to 2M tokens), cognitive focus is not uniform across that space.

### The 'Lost in the Middle' Problem:
LLMs typically prioritize information placed at the absolute **start** and absolute **end** of the context window. Information placed in the middle is significantly more likely to be ignored or hallucinated. 

**Strategy**: Place current instructions and high-priority data at the very end of the prompt sequence.

## 5. Context Distillation (Compression)
When context limits are reached, or when minimizing latency is required, information must be distilled. This involves reducing the volume of data without losing the core signal.

In [3]:
# Example: Incremental Summarization Strategy
def summarize_history(messages):
    summarizer = ChatGoogleGenerativeAI(model=model, temperature=0)
    summary_prompt = f"Summarize the following conversation history into 2 concise sentences, preserving all key decisions and entities:\n{messages}"
    summary = summarizer.invoke(summary_prompt)
    return summary.content

history = [
    HumanMessage(content="My name is John and I want to build a house in Texas."),
    AIMessage(content="Hello John, building in Texas is high-demand. Do you have a budget?"),
    HumanMessage(content="Yes, $500,000.")
]
print(f"Distilled Context: {summarize_history(history)}")

Distilled Context: John wants to build a house in Texas. He has a budget of $500,000 for the project.


## 6. Advanced Patterns: Sliding Windows
Sliding windows ensure the model always has access to the most recent X messages, effectively preventing "memory overflow" in long conversations by pruning older interactions.

In [4]:
def sliding_window_manager(all_messages, window_size=5):
    # Keeping only the most recent N messages to maintain focus and efficiency
    if len(all_messages) > window_size:
        print(f"Window Full ({len(all_messages)} messages). Pruning oldest...")
        return all_messages[-window_size:]
    return all_messages

# Simulating a long sequence
mock_chat = [HumanMessage(content=f"Msg {i}") for i in range(10)]
managed_chat = sliding_window_manager(mock_chat, window_size=3)
print(f"Active Context Messages: {len(managed_chat)}")

Window Full (10 messages). Pruning oldest...
Active Context Messages: 3


## 7. Production Optimization: Gemini Prompt Caching
Prompt Caching is an advanced feature for high-scale applications. It allows the first portion of the context (which often remains static across calls) to be cached in the model's infrastructure.

### Benefits:
- **Reduced Latency**: Faster processing of large system instructions.
- **Cost Savings**: Significant discounts on input tokens for cached portions.

This is particularly effective for large PDF analysis or complex system personas that don't change frequently.

## 8. Real-World Project: Context-Aware Conversation Manager
This final section combines all the previous techniques into a reusable class for managing complex agent state.

In [5]:
class ConversationManager:
    def __init__(self, model_name=model, max_memory=5):
        self.llm = ChatGoogleGenerativeAI(model=model_name)
        self.history = []
        self.max_memory = max_memory
        self.summary = "No prior history."

    def add_message(self, role, content):
        if role == "user":
            self.history.append(HumanMessage(content=content))
        else:
            self.history.append(AIMessage(content=content))
        
        # Automatic maintenance: If history exceeds limit, summarize oldest and prune
        if len(self.history) > self.max_memory:
            self.update_background_summary()

    def update_background_summary(self):
        to_summarize = self.history[:-2] # Keep the very latest interaction
        summary_prompt = f"Summarize the key facts from this interaction history: {to_summarize}"
        self.summary = self.llm.invoke(summary_prompt).content
        self.history = self.history[-2:] # Prune memory to just the last interaction
        print(f"[System] Context compressed. New Summary: {self.summary[:50]}...")

    def get_full_context(self):
        return [
            SystemMessage(content=f"Background Context Summary: {self.summary}"),
            *self.history
        ]

# Demonstration
manager = ConversationManager(max_memory=3)
manager.add_message("user", "My favorite color is Blue.")
manager.add_message("assistant", "I will remember that.")
manager.add_message("user", "I live in NYC.") # Triggering compression on next add
manager.add_message("assistant", "NYC is great!")

full_context = manager.get_full_context()
print(f"\nFinal Compressed Context Structure:\n{full_context}")

[System] Context compressed. New Summary: The human's favorite color is Blue. The AI acknowl...

Final Compressed Context Structure:
[SystemMessage(content="Background Context Summary: The human's favorite color is Blue. The AI acknowledged this information and stated it would remember it.", additional_kwargs={}, response_metadata={}), HumanMessage(content='I live in NYC.', additional_kwargs={}, response_metadata={}), AIMessage(content='NYC is great!', additional_kwargs={}, response_metadata={})]


--- 

## Conclusion
Context Engineering is the foundation for building long-running, autonomous agents. By mastering token computation, sampling, and strategic memory management, applications become more intelligent, more responsive, and significantly more cost-efficient. 